[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/085.%20The%20Uncapacitated%20Facility%20Location%20Problem/P85-Tier-2_executed.ipynb)

---



# 85. The Uncapacitated Facility Location Problem
## Tier 2: The Classic Heuristic (Cross-Entropy Method)

### Purpose and scope
This notebook uses the Cross-Entropy Method (CEM), a probabilistic sampling heuristic, to find a low-cost facility-opening plan. The input is a fixed cost for each facility and a transportation cost for each customer–facility pair. The output is a sampled facility plan, a least-transport-cost assignment of every customer to an open facility, and a record of the search.

The example contains five candidate facilities and eight customers. Its transportation-cost matrix is specified explicitly in the data cell. The method is stochastic: its probability updates are learned from sampled solutions, and it does not certify that the returned plan is globally optimal. No mathematical optimum from another tier is supplied here, so this notebook does not report an optimality gap.

### Model assumptions
- Facilities have no capacity constraints.
- Every customer is assigned to exactly one open facility.
- Opening and transportation costs are deterministic, additive, and expressed in the same unspecified cost units.
- A sampled facility configuration must open at least one facility. If sampling opens none, the facility with the highest current opening probability is opened.
- Given the open facilities, each customer is assigned to a facility with minimum transportation cost. If several open facilities tie, the first one in index order is selected.
- Opening decisions are sampled independently using one Bernoulli probability per facility. The probabilities are updated from the elite sample, with smoothing and clipping.

The model does not represent facility capacities, service-level requirements, uncertainty in costs, or constraints on which customers may use which facilities.

### Mathematical model
Let $F=\{1,\ldots,n_F\}$ be the set of candidate facilities and $C=\{1,\ldots,n_C\}$ the set of customers. Let $f_j$ be the fixed cost of opening facility $j$, and let $c_{ij}$ be the transportation cost from facility $j$ to customer $i$. Define binary decision variables $y_j$ and $x_{ij}$: $y_j=1$ if facility $j$ is open, and $x_{ij}=1$ if customer $i$ is assigned to facility $j$.

The uncapacitated facility location model is

$$
\begin{aligned}
\min_{x,y}\quad & \sum_{j\in F} f_j y_j + \sum_{i\in C}\sum_{j\in F} c_{ij}x_{ij} \\
\text{subject to}\quad
 & \sum_{j\in F} x_{ij}=1 && \forall i\in C,\\
 & x_{ij}\leq y_j && \forall i\in C,\ j\in F,\\
 & \sum_{j\in F}y_j\geq 1,\\
 & x_{ij},y_j\in\{0,1\} && \forall i\in C,\ j\in F.
\end{aligned}
$$

The first constraint assigns each customer exactly once; the second prevents assignment to a closed facility; and the third ensures that at least one facility is open. In the code, `facilities_open` represents $y$, `assignments` represents the customer choices encoded by $x$, and `fixed_cost`, `transport_cost`, and `total_cost` evaluate the objective. Because facilities have no capacity limits and assignments have no other constraints, the least-cost assignment for a fixed opening plan can be computed independently for each customer by choosing the cheapest open facility.

### Cross-Entropy Method: intuition and update
A facility-opening plan is a binary vector. The method represents its sampling distribution by probabilities $p_j$, one for each facility. Initially, every facility has probability $0.5$ of opening. Each iteration samples $N$ plans, evaluates them, and selects the lowest-cost fraction as the elite set. Facilities appearing in more elite plans receive larger empirical opening probabilities. Smoothing blends those empirical values with the previous probabilities so that learning proceeds gradually.

Formally, for a sampled plan $Y=(Y_1,\ldots,Y_{n_F})$, the independent Bernoulli model is

$$
\Pr(Y=y)=\prod_{j\in F}p_j^{y_j}(1-p_j)^{1-y_j}.
$$

If $E_t$ is the elite set at iteration $t$, the empirical opening probability and the smoothed update are

$$
\widehat p_{j,t}=\frac{1}{|E_t|}\sum_{s\in E_t}y_j^{(s)},
\qquad
p_{j,t+1}=\operatorname{clip}_{[0.01,\,0.99]}\!\left(\alpha\widehat p_{j,t}+(1-\alpha)p_{j,t}\right).
$$

Here, $y_j^{(s)}$ is the opening decision for facility $j$ in elite sample $s$, and $\alpha$ is the smoothing parameter. Clipping keeps probabilities away from zero and one, so facilities remain possible to sample throughout the search. The code implements the empirical frequency in `update_probabilities`, followed by smoothing and `np.clip`.

The base state is the initial probability vector, with every entry equal to $0.5$. The iteration maps the current probabilities to sampled plans, an elite set, and updated probabilities; it stops when the largest absolute probability change is below `tol` or when `max_iter` iterations have been completed. This is an iterative stochastic procedure, not a Bellman recursion. With $T$ iterations, $N$ samples per iteration, $n_F$ facilities, and $n_C$ customers, the dominant work is $O(TN(n_F+n_Cn_F))$. Storing a sample population takes $O(N(n_F+n_C))$ space; the recorded probability history takes $O(Tn_F)$ space.

### Procedure
1. Initialize an independent opening probability for every facility.
2. Sample facility configurations and repair any all-closed configuration by opening the facility with the highest probability.
3. Assign each customer to its cheapest open facility and calculate the fixed, transportation, and total costs.
4. Select the lowest-cost $\rho$ fraction of the samples as the elite set.
5. Update the probabilities from elite opening frequencies, using smoothing parameter $\alpha$.
6. Continue until the probability-change tolerance is met or the iteration limit is reached; retain the best sampled solution.

### Example parameters and interpretation
The example uses fixed costs `[80, 120, 95, 110, 85]`, the specified transportation-cost matrix, and CEM parameters $N=500$, $\rho=0.2$, and $\alpha=0.8$. Here, $N$ is the sample size per iteration, $\rho$ is the elite fraction, and $\alpha$ controls the weight given to the latest elite frequencies. The final probabilities describe the search distribution; they are not statistical confidence levels and do not guarantee that a facility should be opened in an optimal plan.

The recorded probability vector for each iteration is the distribution used to generate that iteration's samples. The final vector printed after solving is the distribution after the last update. Cost and probability plots should be read as diagnostics of this particular stochastic run, not as an optimality certificate.

In [ ]:
from dataclasses import dataclass
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np

# A fixed seed makes the sampling sequence reproducible for a given execution order.
np.random.seed(85)

print("Libraries imported successfully for the Cross-Entropy Method.")

In [ ]:
@dataclass
class UFLPInstance:
    """Data for an uncapacitated facility location problem."""

    n_facilities: int
    n_customers: int
    fixed_costs: List[float]
    transport_costs: np.ndarray

    def __post_init__(self):
        """Check dimensions and finite, nonnegative cost data."""
        if self.n_facilities <= 0 or self.n_customers <= 0:
            raise ValueError("The instance must have at least one facility and one customer.")
        if len(self.fixed_costs) != self.n_facilities:
            raise ValueError("The number of fixed costs must match the number of facilities.")
        if self.transport_costs.shape != (self.n_customers, self.n_facilities):
            raise ValueError("Transportation costs must have shape (n_customers, n_facilities).")
        if not np.all(np.isfinite(self.fixed_costs)) or not np.all(np.isfinite(self.transport_costs)):
            raise ValueError("All costs must be finite.")
        if any(cost < 0 for cost in self.fixed_costs) or np.any(self.transport_costs < 0):
            raise ValueError("This example requires nonnegative costs.")


@dataclass
class UFLPSolution:
    """A facility-opening plan, its assignments, and its cost components."""

    facilities_open: List[bool]
    assignments: List[int]
    total_cost: float
    fixed_cost: float
    transport_cost: float
    feasible: bool


@dataclass
class CEParameters:
    """Parameters controlling the Cross-Entropy search."""

    N: int = 500
    rho: float = 0.2
    alpha: float = 0.8
    max_iter: int = 50
    tol: float = 1e-6

    def __post_init__(self):
        """Reject parameter values that cannot define a valid iteration."""
        if self.N <= 0:
            raise ValueError("N must be a positive integer.")
        if not 0 < self.rho <= 1:
            raise ValueError("rho must be in the interval (0, 1].")
        if not 0 < self.alpha <= 1:
            raise ValueError("alpha must be in the interval (0, 1].")
        if self.max_iter <= 0:
            raise ValueError("max_iter must be a positive integer.")
        if self.tol < 0:
            raise ValueError("tol must be nonnegative.")


print("Problem data structures and parameter checks are ready.")

In [ ]:
def create_ce_example_instance() -> UFLPInstance:
    """Create the specified five-facility, eight-customer example."""
    n_facilities = 5
    n_customers = 8
    fixed_costs = [80, 120, 95, 110, 85]

    # The seeded draw is retained as part of the example's reproducible setup;
    # the explicitly listed matrix below defines the actual instance data.
    np.random.seed(42)
    transport_costs = np.random.randint(10, 50, (n_customers, n_facilities))

    transport_costs[0] = [12, 35, 28, 18, 32]
    transport_costs[1] = [25, 15, 22, 30, 18]
    transport_costs[2] = [18, 28, 14, 25, 20]
    transport_costs[3] = [30, 12, 25, 15, 28]
    transport_costs[4] = [22, 18, 30, 12, 25]
    transport_costs[5] = [15, 25, 18, 28, 22]
    transport_costs[6] = [28, 22, 15, 25, 18]
    transport_costs[7] = [20, 30, 22, 18, 15]

    return UFLPInstance(
        n_facilities=n_facilities,
        n_customers=n_customers,
        fixed_costs=fixed_costs,
        transport_costs=transport_costs,
    )


ce_instance = create_ce_example_instance()

print("Cross-Entropy UFLP instance:")
print(f"  Facilities: {ce_instance.n_facilities}")
print(f"  Customers: {ce_instance.n_customers}")
print(f"  Fixed costs: {ce_instance.fixed_costs}")
print(f"  Transportation-cost matrix shape: {ce_instance.transport_costs.shape}")

print("\nTransportation-cost matrix:")
print("Customer\t" + "\t".join(f"F{j + 1}" for j in range(ce_instance.n_facilities)))
for i in range(ce_instance.n_customers):
    row = "\t".join(str(value) for value in ce_instance.transport_costs[i])
    print(f"C{i + 1}\t{row}")

In [ ]:
class CrossEntropyMethod:
    """Cross-Entropy search for the uncapacitated facility location problem."""

    def __init__(self, instance: UFLPInstance, params: CEParameters):
        self.instance = instance
        self.params = params
        self.probabilities = np.full(instance.n_facilities, 0.5, dtype=float)
        self.best_solution: Optional[UFLPSolution] = None
        self.history = {
            "best_costs": [],
            "avg_costs": [],
            "probabilities": [],
            "elite_costs": [],
        }

    def generate_sample_solution(self) -> UFLPSolution:
        """Sample openings, assign customers, and evaluate one feasible plan."""
        facilities_open = [
            np.random.random() < self.probabilities[j]
            for j in range(self.instance.n_facilities)
        ]

        # The model requires an open facility so every customer can be assigned.
        if not any(facilities_open):
            facilities_open[int(np.argmax(self.probabilities))] = True

        assignments = []
        for i in range(self.instance.n_customers):
            open_facilities = [
                j for j, is_open in enumerate(facilities_open) if is_open
            ]
            best_facility = min(
                open_facilities,
                key=lambda j: self.instance.transport_costs[i, j],
            )
            assignments.append(best_facility)

        fixed_cost = sum(
            self.instance.fixed_costs[j]
            for j, is_open in enumerate(facilities_open)
            if is_open
        )
        transport_cost = sum(
            self.instance.transport_costs[i, assignments[i]]
            for i in range(self.instance.n_customers)
        )
        total_cost = fixed_cost + transport_cost

        return UFLPSolution(
            facilities_open=facilities_open,
            assignments=assignments,
            total_cost=total_cost,
            fixed_cost=fixed_cost,
            transport_cost=transport_cost,
            feasible=True,
        )

    def generate_samples(self, N: int) -> List[UFLPSolution]:
        """Generate a population of N independently sampled plans."""
        return [self.generate_sample_solution() for _ in range(N)]

    def select_elite(self, samples: List[UFLPSolution]) -> List[UFLPSolution]:
        """Return the lowest-cost rho fraction of a nonempty sample population."""
        if not samples:
            raise ValueError("At least one sample is required to select an elite set.")

        sorted_samples = sorted(samples, key=lambda solution: solution.total_cost)
        n_elite = max(1, int(len(sorted_samples) * self.params.rho))
        return sorted_samples[:n_elite]

    def update_probabilities(self, elite: List[UFLPSolution]) -> np.ndarray:
        """Estimate elite opening frequencies and apply the smoothed update."""
        if not elite:
            raise ValueError("The elite set must not be empty.")

        new_probabilities = np.mean(
            np.asarray([solution.facilities_open for solution in elite], dtype=float),
            axis=0,
        )
        smoothed_probabilities = (
            self.params.alpha * new_probabilities
            + (1 - self.params.alpha) * self.probabilities
        )

        # Clipping preserves a nonzero chance of sampling every opening decision.
        return np.clip(smoothed_probabilities, 0.01, 0.99)

    def check_convergence(
        self,
        old_probabilities: np.ndarray,
        new_probabilities: np.ndarray,
    ) -> bool:
        """Test whether the largest probability change is below the tolerance."""
        max_change = np.max(np.abs(new_probabilities - old_probabilities))
        return max_change < self.params.tol

    def solve(self, verbose: bool = True) -> UFLPSolution:
        """Run the iterative sampling, elite-selection, and update procedure."""
        if verbose:
            print("Starting the Cross-Entropy Method for UFLP.")
            print(
                f"Parameters: N={self.params.N}, rho={self.params.rho}, "
                f"alpha={self.params.alpha}"
            )

        for iteration in range(self.params.max_iter):
            samples = self.generate_samples(self.params.N)
            costs = [solution.total_cost for solution in samples]
            avg_cost = float(np.mean(costs))
            best_cost = min(costs)

            elite = self.select_elite(samples)
            elite_avg = float(np.mean([solution.total_cost for solution in elite]))

            current_best = elite[0]
            if (
                self.best_solution is None
                or current_best.total_cost < self.best_solution.total_cost
            ):
                self.best_solution = current_best

            # Store the distribution that generated this iteration's population.
            self.history["best_costs"].append(best_cost)
            self.history["avg_costs"].append(avg_cost)
            self.history["probabilities"].append(self.probabilities.copy())
            self.history["elite_costs"].append(elite_avg)

            if verbose and ((iteration + 1) % 5 == 0 or iteration == 0):
                print(
                    f"Iteration {iteration + 1}: best cost = {best_cost:.1f}, "
                    f"elite average = {elite_avg:.1f}, "
                    f"population average = {avg_cost:.1f}"
                )

            old_probabilities = self.probabilities.copy()
            self.probabilities = self.update_probabilities(elite)

            if self.check_convergence(old_probabilities, self.probabilities):
                if verbose:
                    print(f"Probability convergence reached at iteration {iteration + 1}.")
                break

        if verbose:
            print(f"Completed {iteration + 1} iteration(s).")

        if self.best_solution is None:
            raise RuntimeError("The search did not produce a solution.")
        return self.best_solution


print("The Cross-Entropy search procedure is ready.")

In [ ]:
ce_params = CEParameters(N=500, rho=0.2, alpha=0.8, max_iter=50)
ce_solver = CrossEntropyMethod(ce_instance, ce_params)
ce_solution = ce_solver.solve()

print("\nCross-Entropy solution results")
print(f"Total cost: {ce_solution.total_cost:.2f}")
print(f"Fixed cost: {ce_solution.fixed_cost:.2f}")
print(f"Transportation cost: {ce_solution.transport_cost:.2f}")
print(
    "Facilities opened: "
    f"{[j + 1 for j, is_open in enumerate(ce_solution.facilities_open) if is_open]}"
)
print(f"Number of facilities open: {sum(ce_solution.facilities_open)}")

print("\nCustomer assignments:")
for i, facility in enumerate(ce_solution.assignments):
    cost = ce_instance.transport_costs[i, facility]
    print(f"  Customer {i + 1} -> Facility {facility + 1} (transport cost: {cost})")

print(f"\nFinal opening probabilities: {ce_solver.probabilities}")

In [ ]:
def visualize_ce_results(
    ce_solver: CrossEntropyMethod,
    solution: UFLPSolution,
) -> None:
    """Plot search history, opening decisions, and the returned cost breakdown."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Cross-Entropy Method: UFLP Solution Analysis",
        fontsize=16,
        fontweight="bold",
    )

    iterations = np.arange(1, len(ce_solver.history["best_costs"]) + 1)

    ax1 = axes[0, 0]
    ax1.plot(
        iterations,
        ce_solver.history["best_costs"],
        "o-",
        linewidth=2,
        markersize=5,
        label="Best sample cost",
    )
    ax1.plot(
        iterations,
        ce_solver.history["elite_costs"],
        "s--",
        linewidth=2,
        markersize=5,
        label="Elite average cost",
    )
    ax1.plot(
        iterations,
        ce_solver.history["avg_costs"],
        "d:",
        linewidth=2,
        markersize=5,
        label="Population average cost",
    )
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Cost (unspecified cost units)")
    ax1.set_title("Sampled Cost History")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    probability_history = np.asarray(ce_solver.history["probabilities"]).T
    for j in range(ce_solver.instance.n_facilities):
        ax2.plot(
            iterations,
            probability_history[j],
            "o-",
            linewidth=2,
            markersize=4,
            label=f"Facility {j + 1}",
        )
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Opening probability used for sampling")
    ax2.set_title("Probability Distribution History")
    ax2.set_ylim(0, 1)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    facility_labels = [f"F{j + 1}" for j in range(ce_solver.instance.n_facilities)]
    opening_status = np.asarray(solution.facilities_open, dtype=int)
    bar_colors = ["#F18F01" if is_open else "#C73E1D" for is_open in solution.facilities_open]
    bars = ax3.bar(facility_labels, opening_status, color=bar_colors)
    ax3.set_title("Facility Opening Decisions")
    ax3.set_xlabel("Facility")
    ax3.set_ylabel("Opening decision (1 = open, 0 = closed)")
    ax3.set_ylim(-0.1, 1.1)
    for j, bar in enumerate(bars):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            0.5,
            f"Fixed cost: {ce_solver.instance.fixed_costs[j]}\n"
            f"Probability: {ce_solver.probabilities[j]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )

    ax4 = axes[1, 1]
    cost_values = [solution.fixed_cost, solution.transport_cost]
    cost_labels = ["Fixed cost", "Transportation cost"]
    ax4.pie(
        cost_values,
        labels=cost_labels,
        colors=["#2E86AB", "#A23B72"],
        autopct="%1.1f%%",
        startangle=90,
    )
    ax4.set_title(f"Cost Breakdown\nTotal: {solution.total_cost:.2f} cost units")

    plt.tight_layout()
    plt.show()

    print("\nOpening-probability interpretation")
    print("These probabilities describe the final sampling distribution; they are not confidence levels.")
    for j, probability in enumerate(ce_solver.probabilities):
        decision = "OPEN" if solution.facilities_open[j] else "CLOSED"
        print(
            f"  Facility {j + 1}: probability = {probability:.3f}; "
            f"returned plan decision = {decision}"
        )


visualize_ce_results(ce_solver, ce_solution)

In [ ]:
def parameter_sensitivity_analysis(instance: UFLPInstance) -> None:
    """Examine how the specified CEM parameters affect sampled outcomes."""
    print("Parameter sensitivity analysis")

    print("\n1. Sample size (N):")
    sample_sizes = [100, 300, 500, 700, 1000]
    n_results = []
    for N in sample_sizes:
        params = CEParameters(N=N, rho=0.2, alpha=0.8, max_iter=30)
        solver = CrossEntropyMethod(instance, params)
        solution = solver.solve(verbose=False)
        result = {
            "N": N,
            "cost": solution.total_cost,
            "facilities": sum(solution.facilities_open),
            "iterations": len(solver.history["best_costs"]),
        }
        n_results.append(result)
        print(
            f"  N={N}: cost={result['cost']:.1f}, "
            f"facilities open={result['facilities']}, "
            f"iterations={result['iterations']}"
        )

    print("\n2. Elite fraction (rho):")
    rho_values = [0.1, 0.15, 0.2, 0.25, 0.3]
    rho_results = []
    for rho in rho_values:
        params = CEParameters(N=500, rho=rho, alpha=0.8, max_iter=30)
        solver = CrossEntropyMethod(instance, params)
        solution = solver.solve(verbose=False)
        result = {
            "rho": rho,
            "cost": solution.total_cost,
            "facilities": sum(solution.facilities_open),
            "iterations": len(solver.history["best_costs"]),
        }
        rho_results.append(result)
        print(
            f"  rho={rho:.2f}: cost={result['cost']:.1f}, "
            f"facilities open={result['facilities']}, "
            f"iterations={result['iterations']}"
        )

    print("\n3. Smoothing parameter (alpha):")
    alpha_values = [0.5, 0.6, 0.7, 0.8, 0.9]
    alpha_results = []
    for alpha in alpha_values:
        params = CEParameters(N=500, rho=0.2, alpha=alpha, max_iter=30)
        solver = CrossEntropyMethod(instance, params)
        solution = solver.solve(verbose=False)
        result = {
            "alpha": alpha,
            "cost": solution.total_cost,
            "facilities": sum(solution.facilities_open),
            "iterations": len(solver.history["best_costs"]),
        }
        alpha_results.append(result)
        print(
            f"  alpha={alpha:.1f}: cost={result['cost']:.1f}, "
            f"facilities open={result['facilities']}, "
            f"iterations={result['iterations']}"
        )

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    analyses = [
        (axes[0], n_results, "N", "Sample size (N)", "Sample Size Sensitivity"),
        (axes[1], rho_results, "rho", "Elite fraction (rho)", "Elite Fraction Sensitivity"),
        (axes[2], alpha_results, "alpha", "Smoothing parameter (alpha)", "Smoothing Sensitivity"),
    ]

    for ax, results, parameter, x_label, title in analyses:
        x_values = [result[parameter] for result in results]
        costs = [result["cost"] for result in results]
        facility_counts = [result["facilities"] for result in results]

        cost_line = ax.plot(
            x_values,
            costs,
            "o-",
            linewidth=2,
            markersize=7,
            color="#2E86AB",
            label="Best sampled cost",
        )
        ax.set_xlabel(x_label)
        ax.set_ylabel("Best sampled cost (cost units)", color="#2E86AB")
        ax.tick_params(axis="y", labelcolor="#2E86AB")
        ax.set_title(title)
        ax.grid(True, alpha=0.3)

        facility_axis = ax.twinx()
        facility_line = facility_axis.plot(
            x_values,
            facility_counts,
            "s--",
            linewidth=2,
            markersize=7,
            color="#A23B72",
            label="Facilities open",
        )
        facility_axis.set_ylabel("Number of facilities open", color="#A23B72")
        facility_axis.tick_params(axis="y", labelcolor="#A23B72")
        ax.legend(cost_line + facility_line, ["Best sampled cost", "Facilities open"], loc="best")

    plt.tight_layout()
    plt.show()


parameter_sensitivity_analysis(ce_instance)

In [ ]:
def validate_and_report_ce_solution(
    instance: UFLPInstance,
    solution: UFLPSolution,
) -> None:
    """Check feasibility and reconcile the returned solution's cost components."""
    if len(solution.facilities_open) != instance.n_facilities:
        raise AssertionError("The opening vector has the wrong number of facilities.")
    if len(solution.assignments) != instance.n_customers:
        raise AssertionError("The assignment vector has the wrong number of customers.")
    if not any(solution.facilities_open):
        raise AssertionError("At least one facility must be open.")

    for i, facility in enumerate(solution.assignments):
        if facility < 0 or facility >= instance.n_facilities:
            raise AssertionError(f"Customer {i + 1} has an invalid facility assignment.")
        if not solution.facilities_open[facility]:
            raise AssertionError(f"Customer {i + 1} is assigned to a closed facility.")

    expected_fixed_cost = sum(
        instance.fixed_costs[j]
        for j, is_open in enumerate(solution.facilities_open)
        if is_open
    )
    expected_transport_cost = sum(
        instance.transport_costs[i, solution.assignments[i]]
        for i in range(instance.n_customers)
    )
    expected_total_cost = expected_fixed_cost + expected_transport_cost

    if not np.isclose(solution.fixed_cost, expected_fixed_cost):
        raise AssertionError("The reported fixed cost does not match the opening decisions.")
    if not np.isclose(solution.transport_cost, expected_transport_cost):
        raise AssertionError("The reported transportation cost does not match the assignments.")
    if not np.isclose(solution.total_cost, expected_total_cost):
        raise AssertionError("The reported total cost does not equal its cost components.")

    print("Solution feasibility and cost checks passed.")
    print(f"Total cost: {solution.total_cost:.2f} cost units")
    print(f"Fixed cost: {solution.fixed_cost:.2f} cost units")
    print(f"Transportation cost: {solution.transport_cost:.2f} cost units")
    print(
        "Facilities open: "
        f"{[j + 1 for j, is_open in enumerate(solution.facilities_open) if is_open]}"
    )
    print(
        "The reported plan is the best sampled plan retained by this run; "
        "the checks establish feasibility and cost consistency, not global optimality."
    )


validate_and_report_ce_solution(ce_instance, ce_solution)

## Strengths, limitations, and interpretation

The Cross-Entropy Method learns a sampling distribution from elite solutions rather than evaluating every possible facility subset. For $n_F$ facilities, there are $2^{n_F}-1$ nonempty opening configurations, while this implementation evaluates $N$ sampled configurations per iteration. Its computational effort therefore depends on the chosen sample size and iteration limit, rather than explicitly enumerating all configurations. This does not make the method a polynomial-time guarantee for finding an optimum: the quality of a finite run depends on sampling, parameter choices, and the instance.

### Strengths
- Elite samples provide a systematic feedback signal for updating facility-opening probabilities.
- The probability vector gives a useful summary of which opening decisions were common among the elite plans during the search.
- For a fixed opening configuration, customer assignments are evaluated directly by selecting the cheapest open facility.
- Sampling and evaluation are conceptually separable across candidate plans, although this notebook executes them sequentially.

### Limitations
- A heuristic run does not certify global optimality, and its best sampled plan can depend on the random sequence.
- The sample size, elite fraction, smoothing parameter, stopping tolerance, and iteration limit affect the search.
- The stopping rule measures changes in the probability vector, not the distance from a known optimum or a proof of solution quality.
- Clipping keeps opening probabilities between 0.01 and 0.99, so the distribution does not become fully deterministic.
- The reported probabilities are sampling probabilities, not confidence levels or probabilities that a facility belongs to a globally optimal plan.
- The sensitivity experiment performs separate stochastic runs in sequence; observed differences combine parameter effects with sampling variation.

Use the cost history, final plan, probability history, and sensitivity plots as evidence about the behavior of this run. A low sampled cost is a feasible candidate, not an optimality certificate. The mathematical model and assumptions determine what the plan means; the heuristic determines how candidate plans are explored.